# C25 — Évaluation hors domaine des modèles d'émotion sur CaFE (H2, P7, E1)

**Évaluation seulement** : aucune adaptation à CaFE. Les 5 modèles entraînés sur CREMA-D + RAVDESS
(CNN, CRNN, têtes sur wav2vec 2.0, mHuBERT-147 et Whisper base) sont appliqués tels quels aux
792 extraits de CaFE (français du Québec, 12 acteurs, surprise exclue).

**H2** : « mHuBERT dépasse wav2vec 2.0 anglais sur le français ». Test principal : écart de F1
macro mHuBERT − wav2vec 2.0 par bootstrap **apparié sur les acteurs** (12 acteurs, 1 000 tirages,
graine 42). Complément : McNemar exact sur les extraits (il suppose des extraits indépendants,
ce qui est faux ici : un acteur joue 78 extraits ; il ne sert que de repère). Verdict : confirmée si
l'intervalle de l'écart est entièrement positif, infirmée s'il est entièrement négatif, non
tranchée sinon.

**Avant CaFE** : chaque modèle doit retrouver son F1 macro de test en domaine (graine 42) à ±0,01
près. Pour des poids relus (Hugging Face, sortie de C11), un écart signale une chaîne différente
de celle de l'entraînement : le notebook s'arrête. Un modèle réentraîné ici (CNN, têtes C11
absentes) peut s'écarter un peu, le calcul sur GPU n'étant pas exactement reproductible :
l'écart est affiché et à noter dans le rapport, sans arrêt.

**Chute** : toujours sur le même réseau, F1 de test en domaine mesuré ici moins F1 sur CaFE.

Où sont les poids (C9, C11, C12) :
- **CRNN** et **tête Whisper** : Hugging Face, à la révision figée de `configs/registry.yaml` ;
- **têtes wav2vec 2.0 et mHuBERT** : sortie Kaggle de C11 (`head_seed42.pt` et `standardization.npz`)
  si elle est attachée au notebook ; sinon réentraînement avec la même configuration et la
  même graine (section f) ;
- **CNN** : jamais sauvegardé (C12 n'a écrit que le meilleur réseau, le CRNN) → réentraînement
  avec la même configuration et la même graine (section g).

Internet activé, GPU T4. Standardisation et température : celles apprises sur l'anglais (R4) ;
rien n'est réestimé sur CaFE.

## a. Démarrage

`BRANCH` : la branche du dépôt à installer (`main`).

In [ ]:
import sys

# Branche installée : main (la branche de la PR disparaît une fois fusionnée).
BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et GPU

In [ ]:
import shutil
import time

import torch

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
NOTEBOOK_START = time.time()
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4.")
device = torch.device("cuda")
print("GPU :", torch.cuda.get_device_name(0))

## c. Jeux de données

CREMA-D et RAVDESS (datasets Kaggle attachés, comme C10 et C12) et CaFE, version 48 kHz (archive
officielle Zenodo, comme C16). Liens dans `/kaggle/working/datasets/<id>`.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
DOWNLOAD_DIR = Path("/kaggle/working/downloads")
CAFE_ZIP_URL = "https://zenodo.org/records/1478765/files/CaFE_48k.zip?download=1"
CREMAD_EXPECTED_WAV = 7442
RAVDESS_EXPECTED_WAV = 1440


def find_cremad_root():
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        if len(list(audio_folder.glob("*.wav"))) == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    raise RuntimeError("CREMA-D introuvable : attacher le dataset ejlok1/cremad.")


def find_ravdess_root():
    for candidate in sorted({actor.parent for actor in INPUT_DIR.glob("**/Actor_01")}):
        if len(list(candidate.glob("Actor_*/*.wav"))) == RAVDESS_EXPECTED_WAV:
            return candidate
    raise RuntimeError("RAVDESS introuvable : attacher uwrfkaggler/ravdess-emotional-speech-audio.")


def download_cafe():
    archive_path = DOWNLOAD_DIR / "cafe.zip"
    extract_dir = DOWNLOAD_DIR / "cafe"
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CAFE_ZIP_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    archive_path.unlink()
    return extract_dir


roots = {"cremad": find_cremad_root(), "ravdess": find_ravdess_root(), "cafe": download_cafe()}
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
for dataset_id, root in roots.items():
    link = DATASETS_DIR / dataset_id
    if link.is_symlink():
        link.unlink()
    link.symlink_to(root)
    print(dataset_id, "→", root)

## d. Configuration, extraits d'émotion et CaFE

Paramètres : `configs/ood.yaml` (modèles, tolérance ±0,01, paire de H2) et
`configs/evaluation.yaml` (1 000 tirages, 95 %, graine 42). Les extraits en domaine sont ceux de C11
et C12 (`emotion_examples` : découpage par acteur P1, 6 classes P2). Pour CaFE, le même
`emotion_examples` : jeu réservé au test (P7), surprise exclue par `configs/labels.yaml`.

In [ ]:
import json

import numpy as np

from echolens.config import CONFIG_DIR, REPO_ROOT
from echolens.core.data.emotions import load_emotion_labels
from echolens.core.data.manifest import audio_path, load_manifest
from echolens.core.data.splits import check_no_speaker_leak, load_speaker_splits, load_split_config
from echolens.training.evaluation import EvaluationConfig
from echolens.training.heads import emotion_examples, load_heads_config
from echolens.training.ood_eval import load_ood_config

ood = load_ood_config()
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
heads_config = load_heads_config()
labels = load_emotion_labels()
class_names = list(labels.emotions)
class_count = len(class_names)
split_config = load_split_config()
speaker_splits = load_speaker_splits(REPO_ROOT / "data" / "splits" / "emotion_split.csv")
check_no_speaker_leak(speaker_splits)
manifests_dir = REPO_ROOT / "data" / "manifests"

# Étape 1 : CaFE, 792 extraits gardés et 12 acteurs (contrôle de l'énoncé de C25).
cafe_all_rows = load_manifest(manifests_dir / "cafe.csv")
cafe_examples = emotion_examples(cafe_all_rows, labels, speaker_splits, split_config)
cafe_rows = [cafe_all_rows[example.position] for example in cafe_examples]
cafe_targets = np.array([example.target for example in cafe_examples], dtype=np.int_)
cafe_actors = [example.speaker for example in cafe_examples]
cafe_genders = [example.gender for example in cafe_examples]
cafe_intensities = [ood.intensity_names[row.intensity] for row in cafe_rows]
# Le neutre de CaFE n'a pas d'intensité : il est écarté de la comparaison faible / forte.
neutral_target = class_names.index(ood.neutral_class)
if len(cafe_rows) != ood.cafe_expected_clips or len(set(cafe_actors)) != ood.cafe_expected_actors:
    raise RuntimeError(f"CaFE : {len(cafe_rows)} extraits, {len(set(cafe_actors))} acteurs")
missing = [row.path for row in cafe_rows if not audio_path(row, DATASETS_DIR).exists()]
if missing:
    raise RuntimeError(f"CaFE : {len(missing)} audios absents, par exemple {missing[:3]}")
print("CaFE :", len(cafe_rows), "extraits,", len(set(cafe_actors)), "acteurs")
print("Classes :", np.bincount(cafe_targets, minlength=class_count).tolist(), class_names)

# Étape 2 : extraits en domaine (CREMA-D et RAVDESS), avec leur partie (P1).
in_domain_rows = []
in_domain_examples = []
for dataset_id in heads_config.emotion_datasets:
    rows = load_manifest(manifests_dir / f"{dataset_id}.csv")
    for example in emotion_examples(rows, labels, speaker_splits, split_config):
        in_domain_rows.append(rows[example.position])
        in_domain_examples.append(example)
split_of_clip = np.array([example.split for example in in_domain_examples])
in_domain_targets = np.array([example.target for example in in_domain_examples], dtype=np.int_)
print({name: int((split_of_clip == name).sum()) for name in ["train", "validation", "test"]})

## e. Chiffres publiés (C9, C11, C12) et fonctions communes

Pour chaque modèle : F1 macro de test **de la graine 42** (celle des poids) et moyenne des 3 graines
(le chiffre cité dans le rapport). La vérification ±0,01 utilise la graine 42 ; la « chute » se
mesure sur le même réseau (son F1 de test en domaine mesuré ici) ; la moyenne publiée
reste affichée.

In [ ]:
from echolens.training.evaluation import (
    accuracy,
    apply_temperature,
    expected_calibration_error,
    fit_temperature,
    macro_f1,
    normalized_confusion,
    per_class_f1,
    scores_by_group,
)
from echolens.training.ood_eval import check_in_domain

RESULTS_DIR = REPO_ROOT / "training" / "results"
published = {}
for model_id in ["cnn", "crnn"]:
    content = json.loads(
        (RESULTS_DIR / f"emotions_family_a_{model_id}.json").read_text(encoding="utf-8")
    )
    for seed_run in content["seeds"]:
        if seed_run["seed"] == ood.reference_seed:
            reference = seed_run["test"]
    published[model_id] = {
        "seed42_f1": reference["macro_f1"],
        "mean_f1": content["summary"]["macro_f1"]["mean"],
        "per_class": reference["f1_per_class"],
    }
c11 = json.loads((RESULTS_DIR / "c11_family_b_emotions.json").read_text(encoding="utf-8"))
seed_position = heads_config.seeds.index(ood.reference_seed)
for model_id in ["wav2vec2-base", "mhubert-147", "whisper-base"]:
    result = c11["models"][model_id]
    published[model_id] = {
        "seed42_f1": result["macro_f1"]["values"][seed_position],
        "mean_f1": result["macro_f1"]["mean"],
        "per_class": result["f1_per_class"],
        "dropout": result["chosen"]["setting"]["dropout"],
    }
for model_id, values in published.items():
    print(
        f"{model_id:14s} publié : graine 42 {values['seed42_f1']:.4f}, "
        f"moyenne {values['mean_f1']:.4f}"
    )

# Ce que chaque modèle produit : probabilités brutes en domaine (validation, test) et sur CaFE.
validation_mask = split_of_clip == "validation"
test_mask = split_of_clip == "test"
y_validation = in_domain_targets[validation_mask]
y_test = in_domain_targets[test_mask]
probabilities = {}  # model_id -> {"validation": ..., "test": ..., "cafe": ...}
temperatures = {}  # model_id -> (T, origine)
weights_source = {}  # model_id -> d'où viennent les poids
retrained = set()  # modèles réentraînés dans ce notebook (CNN, têtes C11 absentes)

## f. Famille B : encodeurs gelés et têtes

Les représentations (moyenne de chaque couche) se calculent avec `embed_manifest` de C10, la même
fonction que pour les chiffres publiés. On les calcule pour CaFE et pour les extraits en domaine
nécessaires : validation et test si les têtes sont retrouvées, **tout** (train compris) s'il faut
réentraîner.

In [ ]:
import torch
from huggingface_hub import hf_hub_download

from echolens.core.models.heads import LayerWeightedHead
from echolens.core.models.production_config import CONFIG_FILE, WEIGHTS_FILE, load_production_config
from echolens.core.models.registry import ModelRegistry
from echolens.core.models.storage import load_weights
from echolens.training.embeddings import load_embeddings_config
from echolens.training.encoders import embed_manifest, load_encoder
from echolens.training.head_training import predict_probabilities, run_emotions
from echolens.training.heads import LabeledSet, Standardization, standardize

embeddings_config = load_embeddings_config()
encoder_configs = {encoder.id: encoder for encoder in embeddings_config.encoders}
registry = ModelRegistry.from_yaml(CONFIG_DIR / "registry.yaml")
registry_entries = {entry.id: entry for entry in registry.models}


def find_c11_head(model_id):
    # Sortie Kaggle de C11 : .../emotions/<modèle>/head_seed42.pt et standardization.npz.
    candidates = sorted(INPUT_DIR.glob(f"**/emotions/{model_id}/head_seed{ood.reference_seed}.pt"))
    if candidates:
        return candidates[0].parent
    return None


c11_folders = {model_id: find_c11_head(model_id) for model_id in ["wav2vec2-base", "mhubert-147"]}
must_retrain = [model_id for model_id, folder in c11_folders.items() if folder is None]
print("Têtes C11 retrouvées :", {k: str(v) for k, v in c11_folders.items()})
print("À réentraîner :", must_retrain)
needed = np.ones(len(in_domain_rows), dtype=bool)
if not must_retrain:
    needed = validation_mask | test_mask

In [ ]:
# Étape 1 : représentations de chaque encodeur, pour CaFE et pour les extraits en domaine utiles.
needed_positions = np.flatnonzero(needed)
needed_rows = [in_domain_rows[position] for position in needed_positions]
# encodeur -> {"cafe": (792, couches, dim), "in_domain": (extraits utiles, couches, dim)}
embeddings = {}
for encoder_id in ["wav2vec2-base", "mhubert-147", "whisper-base"]:
    encoder = load_encoder(encoder_configs[encoder_id], embeddings_config.sample_rate, device)
    cafe_result = embed_manifest(cafe_rows, DATASETS_DIR, encoder)
    in_domain_result = embed_manifest(needed_rows, DATASETS_DIR, encoder)
    embeddings[encoder_id] = {
        "cafe": cafe_result.embeddings,
        "in_domain": in_domain_result.embeddings,
    }
    print(encoder_id, cafe_result.embeddings.shape, in_domain_result.embeddings.shape)
    del encoder
    torch.cuda.empty_cache()

# Position, parmi les extraits utiles, de chaque partie.
useful_split = split_of_clip[needed_positions]
useful_targets = in_domain_targets[needed_positions]
useful_validation = np.flatnonzero(useful_split == "validation")
useful_test = np.flatnonzero(useful_split == "test")
useful_train = np.flatnonzero(useful_split == "train")

In [ ]:
def head_probabilities(head, stats, features):
    standardized = standardize(features, stats)
    return predict_probabilities(head, standardized)


def new_head(layer_count, dimension, dropout):
    return LayerWeightedHead(layer_count, dimension, heads_config.hidden_size, class_count, dropout)


# Étape 2 : tête Whisper base, depuis Hugging Face (révision figée du registre).
entry = registry_entries["echolens-emotion-whisper-base"]
repo_id = entry.weights.removeprefix("hf://")
whisper_config = load_production_config(
    Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision))
)
whisper_weights = Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision))
shape = whisper_config.head
whisper_head = load_weights(
    new_head(shape.layer_count, shape.dimension, shape.dropout), whisper_weights
)
whisper_stats = Standardization(
    mean=np.asarray(whisper_config.standardization_mean, dtype=np.float64),
    std=np.asarray(whisper_config.standardization_std, dtype=np.float64),
)
heads = {"whisper-base": (whisper_head, whisper_stats)}
temperatures["whisper-base"] = (
    whisper_config.calibration.temperature,
    "registre / config.json (anglais)",
)
weights_source["whisper-base"] = f"Hugging Face {repo_id} @ {entry.revision[:8]}"

# Étape 3 : têtes wav2vec 2.0 et mHuBERT : sortie C11 si attachée.
for model_id, folder in c11_folders.items():
    if folder is None:
        continue
    layer_count, dimension = embeddings[model_id]["cafe"].shape[1:]
    head = new_head(layer_count, dimension, published[model_id]["dropout"])
    head.load_state_dict(
        torch.load(folder / f"head_seed{ood.reference_seed}.pt", map_location="cpu")
    )
    head.eval()
    saved = np.load(folder / "standardization.npz")
    heads[model_id] = (head, Standardization(mean=saved["mean"], std=saved["std"]))
    weights_source[model_id] = f"sortie Kaggle de C11 ({folder})"

# Étape 4 : réentraînement (même configuration, mêmes graines) pour les têtes manquantes.
validation_from_run = {}
for model_id in must_retrain:
    features = embeddings[model_id]["in_domain"]
    train_set = LabeledSet(features[useful_train], useful_targets[useful_train])
    validation_set = LabeledSet(features[useful_validation], useful_targets[useful_validation])
    test_set = LabeledSet(features[useful_test], useful_targets[useful_test])
    run = run_emotions(train_set, validation_set, test_set, heads_config, class_count)
    index = heads_config.seeds.index(ood.reference_seed)
    heads[model_id] = (run.heads[index].model, run.heads[index].stats)
    weights_source[model_id] = "réentraîné ici (configs/heads.yaml, graine 42)"
    retrained.add(model_id)
    print(model_id, "réentraîné : réglage", run.chosen.setting)

In [ ]:
# Étape 5 : probabilités brutes en domaine (validation, test) et sur CaFE, pour les 3 têtes.
for model_id, (head, stats) in heads.items():
    in_domain = embeddings[model_id]["in_domain"]
    probabilities[model_id] = {
        "validation": head_probabilities(head, stats, in_domain[useful_validation]),
        "test": head_probabilities(head, stats, in_domain[useful_test]),
        "cafe": head_probabilities(head, stats, embeddings[model_id]["cafe"]),
    }
    if model_id not in temperatures:
        # T apprise sur la validation ANGLAISE (E2), jamais sur CaFE.
        fitted = fit_temperature(
            y_validation,
            probabilities[model_id]["validation"].astype(np.float64),
            evaluation_config,
        )
        temperatures[model_id] = (fitted, "ajustée sur la validation anglaise (fit_temperature)")
    f1 = macro_f1(y_test, probabilities[model_id]["test"].argmax(axis=1))
    print(
        f"{model_id:14s} F1 test en domaine {f1:.4f} "
        f"(publié graine 42 : {published[model_id]['seed42_f1']:.4f})"
    )

## g. Famille A : CRNN (Hugging Face) et CNN (réentraîné)

Log-Mel de chaque extrait par `echolens.core.features` (R5), fenêtre centrale de 3 s, standardisation
par bande de l'entraînement (R4). Le log-Mel des extraits en domaine est celui de C12 : si sa sortie
`emotion_log_mels.npz` est attachée, on la relit ; sinon on la recalcule. Le **CNN** n'a jamais été
sauvegardé : on le réentraîne avec `run_emotion_architecture` (même configuration, mêmes graines) et on
garde la graine 42 à sa meilleure époque de validation, comme C12.

In [ ]:
from echolens.core.audio import load_audio
from echolens.core.features import FeatureConfig, log_mel_spectrogram
from echolens.core.frame_windows import center_windows, window_frame_count
from echolens.core.models.family_a import SpectrogramCrnn
from echolens.training.family_a import BandStatistics, load_family_a_config
from echolens.training.family_a_emotions import (
    compute_emotion_log_mels,
    load_emotion_log_mels,
    save_emotion_log_mels,
)
from echolens.training.family_a_emotions_training import emotion_parts, run_emotion_architecture
from echolens.training.family_a_models import predict as predict_spectrograms

feature_config = FeatureConfig.from_yaml(CONFIG_DIR / "features.yaml")
family_a_config = load_family_a_config()
window_frames = window_frame_count(family_a_config.emotions.window_s, feature_config)

# Étape 1 : log-Mel des extraits en domaine (réutilise C12 s'il est attaché).
LOG_MELS_PATH = Path("/kaggle/working/emotion_log_mels.npz")
attached = sorted(INPUT_DIR.glob("**/emotion_log_mels.npz"))
if attached:
    log_mel_set = load_emotion_log_mels(attached[0])
    print("Log-Mel de C12 relus :", attached[0])
else:
    rows_by_dataset = {}
    for dataset_id in heads_config.emotion_datasets:
        rows_by_dataset[dataset_id] = load_manifest(manifests_dir / f"{dataset_id}.csv")
    log_mel_set = compute_emotion_log_mels(
        rows_by_dataset, DATASETS_DIR, labels, speaker_splits, split_config, feature_config
    )
    save_emotion_log_mels(log_mel_set, LOG_MELS_PATH)
parts = emotion_parts(log_mel_set, window_frames, feature_config.db_floor)
y_test_a = log_mel_set.targets[parts.test_indices]
y_validation_a = log_mel_set.targets[parts.validation_indices]
# Les deux familles doivent voir exactement les mêmes extraits de test, dans le même ordre.
if not np.array_equal(y_test_a, y_test) or not np.array_equal(y_validation_a, y_validation):
    raise RuntimeError(
        "Les extraits de test ou de validation de la famille A diffèrent de ceux de C11"
    )

# Étape 2 : log-Mel de CaFE, fenêtre centrale de 3 s.
cafe_log_mels = []
cafe_lengths = []
for row in cafe_rows:
    signal = load_audio(audio_path(row, DATASETS_DIR))
    log_mel = log_mel_spectrogram(signal, feature_config)
    cafe_log_mels.append(log_mel)
    cafe_lengths.append(log_mel.shape[1])
longest = max(cafe_lengths)
padded = np.full(
    (len(cafe_log_mels), feature_config.n_mels, longest), feature_config.db_floor, dtype=np.float32
)
for index, log_mel in enumerate(cafe_log_mels):
    padded[index, :, : log_mel.shape[1]] = log_mel
cafe_windows = center_windows(
    padded, np.array(cafe_lengths), window_frames, feature_config.db_floor
)
print("CaFE, fenêtres :", cafe_windows.shape)

In [ ]:
# Étape 3 : CRNN depuis Hugging Face.
entry = registry_entries["echolens-emotion-crnn"]
repo_id = entry.weights.removeprefix("hf://")
crnn_config = load_production_config(
    Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision))
)
crnn = SpectrogramCrnn(class_count, crnn_config.network)
crnn = load_weights(crnn, Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision)))
crnn.to(device)
crnn_stats = BandStatistics(
    mean=np.asarray(crnn_config.standardization_mean, dtype=np.float32),
    std=np.asarray(crnn_config.standardization_std, dtype=np.float32),
)
crnn_in_domain = {
    "validation": predict_spectrograms(
        crnn, parts.validation.log_mels, crnn_stats, family_a_config, device
    ).probabilities,
    "test": predict_spectrograms(
        crnn, parts.test.log_mels, crnn_stats, family_a_config, device
    ).probabilities,
}
crnn_in_domain["cafe"] = predict_spectrograms(
    crnn, cafe_windows, crnn_stats, family_a_config, device
).probabilities
probabilities["crnn"] = crnn_in_domain
temperatures["crnn"] = (crnn_config.calibration.temperature, "registre / config.json (anglais)")
weights_source["crnn"] = f"Hugging Face {repo_id} @ {entry.revision[:8]}"
print("CRNN F1 test en domaine", round(macro_f1(y_test, crnn_in_domain["test"].argmax(axis=1)), 4))

In [ ]:
# Étape 4 : CNN réentraîné (GPU, même configuration et mêmes graines que C12).
start = time.time()
cnn_run = run_emotion_architecture("cnn", parts, class_count, family_a_config, device)
print(
    f"CNN réentraîné en {(time.time() - start) / 60:.1f} min, "
    f"taux retenu {cnn_run.chosen_learning_rate:g}"
)
for seed_run in cnn_run.runs:
    if seed_run.seed == ood.reference_seed:
        cnn_seed = seed_run
cnn_stats = cnn_seed.trained.statistics
cnn_network = cnn_seed.trained.network
probabilities["cnn"] = {
    "validation": cnn_seed.validation.probabilities,
    "test": cnn_seed.test.probabilities,
    "cafe": predict_spectrograms(
        cnn_network, cafe_windows, cnn_stats, family_a_config, device
    ).probabilities,
}
fitted = fit_temperature(
    y_validation, probabilities["cnn"]["validation"].astype(np.float64), evaluation_config
)
temperatures["cnn"] = (fitted, "ajustée sur la validation anglaise (fit_temperature)")
weights_source["cnn"] = "réentraîné ici (configs/family_a.yaml, graine 42)"
retrained.add("cnn")
print(
    "CNN F1 test en domaine",
    round(macro_f1(y_test, probabilities["cnn"]["test"].argmax(axis=1)), 4),
)

## h. Contrôle : chaque modèle retrouve son F1 en domaine (±0,01)

Si un modèle aux **poids relus** est hors tolérance, **le notebook s'arrête** : la chaîne de calcul
n'est pas celle de l'entraînement. Prévenir Luciano avec ce tableau. Un modèle **réentraîné ici**
hors tolérance est seulement signalé : son écart est à noter dans le rapport.

In [ ]:
import pandas as pd

check_rows = []
failures = []
deviations = []
measured_in_domain = {}
for model_spec in ood.models:
    model_id = model_spec.id
    measured = macro_f1(y_test, probabilities[model_id]["test"].argmax(axis=1))
    measured_in_domain[model_id] = measured
    target = published[model_id]["seed42_f1"]
    ok = check_in_domain(measured, target, ood.in_domain_tolerance)
    if not ok and model_id in retrained:
        deviations.append(model_id)
    elif not ok:
        failures.append(model_id)
    check_rows.append(
        {
            "modèle": model_id,
            "F1 test retrouvé": round(measured, 4),
            "publié (graine 42)": round(target, 4),
            "publié (moyenne 3 graines)": round(published[model_id]["mean_f1"], 4),
            "écart": round(measured - target, 4),
            "OK ±0,01": ok,
            "poids": weights_source[model_id],
        }
    )
display(pd.DataFrame(check_rows))
if failures:
    raise RuntimeError(
        f"ARRÊT : F1 en domaine hors tolérance pour {failures}. "
        "Ne pas évaluer sur CaFE ; prévenir Luciano."
    )
if deviations:
    print(f"Réentraînés hors tolérance (écart à noter dans le rapport) : {deviations}")
print("Contrôle passé : on peut évaluer sur CaFE.")

## i. Mesures sur CaFE

Pour chaque modèle : F1 macro, accuracy, matrice de confusion, ECE (avec la température apprise sur
l'anglais, et brute), chute de F1 (même réseau : en domaine moins CaFE), F1 par classe, par intensité (faible,
forte, neutre écarté) et par genre. Intervalles à 95 % par **bootstrap sur les acteurs** : mêmes 1 000 tirages
(graine 42) pour tous les modèles.

In [ ]:
from echolens.training.ood_eval import (
    actor_bootstrap_ci,
    draw_actor_samples,
    f1_drop,
    h2_verdict,
    intensity_scores,
    paired_difference,
)

samples = draw_actor_samples(cafe_actors, evaluation_config)
print(len(samples), "tirages d'acteurs")
predictions = {}
results = {}
for model_spec in ood.models:
    model_id = model_spec.id
    raw = probabilities[model_id]["cafe"].astype(np.float64)
    temperature = temperatures[model_id][0]
    calibrated = apply_temperature(raw, temperature, evaluation_config)
    y_pred = raw.argmax(axis=1)
    predictions[model_id] = y_pred
    f1 = macro_f1(cafe_targets, y_pred)
    low, high = actor_bootstrap_ci(cafe_targets, y_pred, samples, evaluation_config)
    results[model_id] = {
        "family": model_spec.family,
        "weights": weights_source[model_id],
        "in_domain_f1": measured_in_domain[model_id],
        "in_domain_f1_mean": published[model_id]["mean_f1"],
        "in_domain_f1_seed42": published[model_id]["seed42_f1"],
        "macro_f1": f1,
        "macro_f1_ci": [low, high],
        "accuracy": accuracy(cafe_targets, y_pred),
        "f1_drop": f1_drop(measured_in_domain[model_id], f1),
        "temperature": temperature,
        "temperature_origin": temperatures[model_id][1],
        "ece_calibrated": expected_calibration_error(
            cafe_targets, calibrated, evaluation_config
        ).ece,
        "ece_raw": expected_calibration_error(cafe_targets, raw, evaluation_config).ece,
        "f1_per_class": per_class_f1(cafe_targets, y_pred, class_count).tolist(),
        "f1_per_class_in_domain": published[model_id]["per_class"],
        "f1_by_intensity": intensity_scores(cafe_targets, y_pred, cafe_intensities, neutral_target),
        "f1_by_gender": scores_by_group(cafe_targets, y_pred, cafe_genders),
        "confusion": normalized_confusion(cafe_targets, y_pred, class_count).tolist(),
    }
table = []
for model_id, result in results.items():
    table.append(
        {
            "modèle": model_id,
            "F1 en domaine": f"{result['in_domain_f1']:.3f}",
            "F1 CaFE [IC 95 % acteurs]": (
                f"{result['macro_f1']:.3f} "
                f"[{result['macro_f1_ci'][0]:.3f} ; {result['macro_f1_ci'][1]:.3f}]"
            ),
            "chute": f"{result['f1_drop']:+.3f}",
            "accuracy": f"{result['accuracy']:.3f}",
            "ECE": f"{result['ece_calibrated']:.3f}",
        }
    )
display(pd.DataFrame(table))
display(pd.DataFrame({m: r["f1_by_intensity"] for m, r in results.items()}).round(3))
display(pd.DataFrame({m: r["f1_by_gender"] for m, r in results.items()}).round(3))
display(
    pd.DataFrame({m: r["f1_per_class"] for m, r in results.items()}, index=class_names).round(3)
)

## j. Test de H2

Principal : écart de F1 macro **mHuBERT − wav2vec 2.0**, bootstrap apparié sur les acteurs (mêmes
acteurs, mêmes extraits à chaque tirage). Complément : McNemar **exact** sur les extraits ; il suppose
des extraits indépendants, ce qui n'est pas vrai ici (78 extraits par acteur, 12 acteurs) : sa
p-valeur est trop optimiste, elle ne sert que de repère.

In [ ]:
from echolens.training.evaluation import mcnemar_test

first, second = ood.h2.first, ood.h2.second
difference, low, high = paired_difference(
    cafe_targets, predictions[first], predictions[second], samples, evaluation_config
)
verdict = h2_verdict(low, high)
always_exact = evaluation_config.model_copy(update={"mcnemar_exact_below": 10**9})
mcnemar = mcnemar_test(cafe_targets, predictions[first], predictions[second], always_exact)
h2 = {
    "hypothesis": "mHuBERT dépasse wav2vec 2.0 anglais sur le français",
    "difference_f1": difference,
    "difference_ci": [low, high],
    "verdict": verdict,
    "mcnemar_exact": {
        "both_right": mcnemar.both_right,
        f"only_{first}_right": mcnemar.only_a_right,
        f"only_{second}_right": mcnemar.only_b_right,
        "both_wrong": mcnemar.both_wrong,
        "p_value": mcnemar.p_value,
    },
    "mcnemar_note": (
        "McNemar suppose des extraits indépendants : faux ici "
        "(12 acteurs, 78 extraits chacun). Indicatif seulement."
    ),
}
print(
    f"{first} − {second} : écart de F1 {difference:+.3f}, "
    f"IC 95 % [{low:+.3f} ; {high:+.3f}] → H2 {verdict}"
)
print("McNemar exact :", h2["mcnemar_exact"])

## k. Résultats, figures et tableau pour la PR

`c25_cafe_ood.json` (→ `training/results/`) ; la figure en domaine contre CaFE et les matrices de
confusion du meilleur modèle de chaque famille sont dessinées par
`training/scripts/make_report_figures.py`, le script unique des figures du rapport (E8,
→ `report/figures/`), jamais à la main.

In [ ]:
import datetime
import subprocess

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
content = {
    "experiment": "c25_cafe_ood",
    "date": datetime.datetime.now(datetime.UTC).isoformat(),
    "branch": BRANCH,
    "commit": commit,
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "classes": class_names,
    "cafe": {
        "clips": len(cafe_rows),
        "actors": len(set(cafe_actors)),
        "clips_per_class": np.bincount(cafe_targets, minlength=class_count).tolist(),
    },
    "bootstrap": {
        "unit": "actor",
        "draws": evaluation_config.n_bootstrap,
        "seed": evaluation_config.seed,
        "confidence": evaluation_config.confidence,
    },
    "in_domain_check": check_rows,
    "models": results,
    "h2": h2,
    "runtime_s": round(time.time() - NOTEBOOK_START),
}
OUTPUT_DIR = Path("/kaggle/working")
json_path = OUTPUT_DIR / "c25_cafe_ood.json"
text = json.dumps(content, indent=2, ensure_ascii=False, default=float) + "\n"
json_path.write_text(text, encoding="utf-8")
# Le script de figures lit le JSON dans le dépôt cloné : on l'y copie, puis on récupère les PNG.
(Path(CLONE_DIR) / "training" / "results").mkdir(parents=True, exist_ok=True)
(Path(CLONE_DIR) / "training" / "results" / "c25_cafe_ood.json").write_text(text, encoding="utf-8")
!cd {CLONE_DIR} && python training/scripts/make_report_figures.py
figure_names = [
    "fig_cafe_f1.png",
    "c25_cafe_confusion_famille_A.png",
    "c25_cafe_confusion_famille_B.png",
]
for figure_name in figure_names:
    png = Path(CLONE_DIR) / "report" / "figures" / figure_name
    shutil.copy(png, OUTPUT_DIR / figure_name)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

In [ ]:
# Tableau et verdict au format de la description de PR (à copier dans la PR).
lines = [
    "| Modèle | F1 en domaine | F1 CaFE [IC 95 %, acteurs] | Chute |",
    "| --- | --- | --- | --- |",
]
for model_id, result in results.items():
    low_f1, high_f1 = result["macro_f1_ci"]
    lines.append(
        f"| {model_id} | {result['in_domain_f1']:.3f} | "
        f"{result['macro_f1']:.3f} [{low_f1:.3f} ; {high_f1:.3f}] | {result['f1_drop']:+.3f} |"
    )
lines.append("")
lines.append(
    f"**H2 ({verdict})** : mHuBERT − wav2vec 2.0 = {difference:+.3f}, "
    f"IC 95 % [{low:+.3f} ; {high:+.3f}] (bootstrap apparié sur les 12 acteurs)."
)
lines.append("")
lines.append("Classes qui s'effondrent (F1 CaFE < 0,25) :")
for model_id, result in results.items():
    collapsed = []
    for name, value in zip(class_names, result["f1_per_class"], strict=True):
        if value < 0.25:
            collapsed.append(f"{name} ({value:.2f})")
    collapsed_text = "aucune"
    if collapsed:
        collapsed_text = ", ".join(collapsed)
    lines.append(f"- {model_id} : {collapsed_text}")
pr_table = "\n".join(lines)
(OUTPUT_DIR / "c25_pr_table.md").write_text(pr_table + "\n", encoding="utf-8")
print(pr_table)

In [ ]:
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in Path("/kaggle/working").iterdir()))